In [44]:
import os
import cv2
import numpy as np
import random
import math
from tqdm import tqdm

In [45]:
# Fake image generation for training data augmentation
def apply_fake_effects(img):
    img = img.copy()

    # Mild blur
    if random.random() < 0.6:
        k = random.choice([1, 3])
        img = cv2.GaussianBlur(img, (k, k), 0)

    # Slight brightness/contrast
    if random.random() < 0.6:
        alpha = random.uniform(0.85, 1.0)
        beta = random.randint(5, 20)
        img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

    # Controlled noise (FIXED)
    if random.random() < 0.5:
        noise = np.random.normal(0, 5, img.shape).astype(np.float32)
        img = img.astype(np.float32) + noise
        img = np.clip(img, 0, 255).astype(np.uint8)

    # Compression simulation
    if random.random() < 0.5:
        h, w = img.shape[:2]
        scale = random.uniform(0.7, 0.9)
        new_w, new_h = int(w * scale), int(h * scale)

        img = cv2.resize(img, (new_w, new_h))
        img = cv2.resize(img, (w, h))

    return img

# Localized Logo Blur
def blur_random_patch(img):
    h, w = img.shape[:2]

    x = random.randint(w//4, 3*w//4)
    y = random.randint(h//4, 3*h//4)

    patch_w = random.randint(w//8, w//4)
    patch_h = random.randint(h//8, h//4)

    x1 = max(0, x - patch_w//2)
    y1 = max(0, y - patch_h//2)
    x2 = min(w, x + patch_w//2)
    y2 = min(h, y + patch_h//2)

    patch = img[y1:y2, x1:x2]
    patch = cv2.GaussianBlur(patch, (9,9), 0)

    img[y1:y2, x1:x2] = patch
    return img

# Scratch / Line Defects
def add_random_lines(img, max_length=20):
    h, w = img.shape[:2]

    for _ in range(random.randint(1, 3)):
        x1, y1 = random.randint(0, w), random.randint(0, h)
        
        angle = random.uniform(0, 2 * math.pi)
        length = random.randint(5, max_length)
        
        x2 = int(x1 + length * math.cos(angle))
        y2 = int(y1 + length * math.sin(angle))
        
        x2 = max(0, min(w - 1, x2))
        y2 = max(0, min(h - 1, y2))

        color = (random.randint(0,50), random.randint(0,50), random.randint(0,50))
        thickness = random.randint(1, 2)

        cv2.line(img, (x1, y1), (x2, y2), color, thickness)

    return img

# Fake Branding Noise
def add_fake_text(img):
    h, w = img.shape[:2]

    texts = ["XYZ", "COPY", "AAA"]

    text = random.choice(texts)

    x = random.randint(0, w-50)
    y = random.randint(20, h-20)

    cv2.putText(
        img,
        text,
        (x, y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.4,
        (0, 0, 0),
        1,
        cv2.LINE_AA
    )

    return img

# Color Patch Distortion
def color_patch_distortion(img):
    h, w = img.shape[:2]

    # random small region
    x = random.randint(0, w - w//4)
    y = random.randint(0, h - h//4)

    patch_w = random.randint(w//10, w//4)
    patch_h = random.randint(h//10, h//4)

    x2 = min(w, x + patch_w)
    y2 = min(h, y + patch_h)

    patch = img[y:y2, x:x2].astype(np.float32)

    # apply color shift (simulate wrong material / paint)
    patch[:,:,0] *= random.uniform(0.7, 1.3)  # B
    patch[:,:,1] *= random.uniform(0.7, 1.3)  # G
    patch[:,:,2] *= random.uniform(0.7, 1.3)  # R

    patch = np.clip(patch, 0, 255).astype(np.uint8)

    img[y:y2, x:x2] = patch

    return img

# Metal Part Distortion
def metal_part_distortion(img):
    h, w = img.shape[:2]

    # simulate strap area (bottom part usually)
    y_start = random.randint(h//2, 3*h//4)
    y_end = min(h, y_start + random.randint(20, 60))

    x_start = random.randint(0, w//2)
    x_end = min(w, x_start + random.randint(30, 80))

    patch = img[y_start:y_end, x_start:x_end].astype(np.float32)

    # make it slightly different shade (cheap metal mismatch)
    factor = random.uniform(0.6, 1.2)
    patch *= factor

    patch = np.clip(patch, 0, 255).astype(np.uint8)

    img[y_start:y_end, x_start:x_end] = patch

    return img

def uneven_lighting(img):
    h, w = img.shape[:2]

    overlay = np.zeros_like(img, dtype=np.float32)

    for i in range(h):
        intensity = (i / h) * random.uniform(0.5, 1.5)
        overlay[i, :] = intensity * 20

    img = img.astype(np.float32) + overlay
    img = np.clip(img, 0, 255).astype(np.uint8)

    return img

def aspect_ratio_distortion(img):
    h, w = img.shape[:2]

    # Stronger distortion
    scale_w = random.uniform(0.7, 1.3)
    scale_h = random.uniform(0.7, 1.3)

    new_w = int(w * scale_w)
    new_h = int(h * scale_h)

    distorted = cv2.resize(img, (new_w, new_h))
    distorted = cv2.resize(distorted, (w, h))

    return distorted

def elliptical_warp(img):
    h, w = img.shape[:2]

    # Create mesh grid
    map_x = np.zeros((h, w), dtype=np.float32)
    map_y = np.zeros((h, w), dtype=np.float32)

    center_x, center_y = w / 2, h / 2

    for y in range(h):
        for x in range(w):
            dx = (x - center_x) / w
            dy = (y - center_y) / h

            # distort more horizontally or vertically
            factor_x = random.uniform(0.9, 1.2)
            factor_y = random.uniform(0.9, 1.2)

            new_x = center_x + dx * w * factor_x
            new_y = center_y + dy * h * factor_y

            map_x[y, x] = new_x
            map_y[y, x] = new_y

    warped = cv2.remap(img, map_x, map_y, interpolation=cv2.INTER_LINEAR)

    return warped

def create_advanced_fake(img):
    img = img.copy()

    # List of transformations
    transformations = [
        aspect_ratio_distortion,
        blur_random_patch,
        add_random_lines,
        add_fake_text,
        color_patch_distortion,
        metal_part_distortion,
        uneven_lighting
    ]

    # ALWAYS apply base mild effects
    img = apply_fake_effects(img)

    # Apply ONLY 2 random transformations
    chosen_transforms = random.sample(transformations, k=2)

    for transform in chosen_transforms:
        img = transform(img)

    return img

In [46]:
# Generate fake dataset for a given brand
def generate_fake_dataset(brand_path, num_fakes=None):
    genuine_path = os.path.join(brand_path, "genuine")
    fake_path = os.path.join(brand_path, "fake")

    os.makedirs(fake_path, exist_ok=True)

    images = os.listdir(genuine_path)

    if num_fakes is None:
        num_fakes = len(images)

    print(f"Generating {num_fakes} fake images for {brand_path}...")

    for i in tqdm(range(num_fakes)):
        img_name = random.choice(images)
        img_path = os.path.join(genuine_path, img_name)

        img = cv2.imread(img_path)

        if img is None:
            continue

        fake_img = create_advanced_fake(img)

        output_path = os.path.join(fake_path, f"fake_{i}.jpg")
        cv2.imwrite(output_path, fake_img)

In [47]:
# Running for all brands in the dataset
DATASET_PATH = "D:/VIT Personal/TrustFilterAI/ml/counterfeit_detection/data/raw"

for brand in os.listdir(DATASET_PATH):
    brand_path = os.path.join(DATASET_PATH, brand)

    if not os.path.isdir(brand_path):
        continue

    generate_fake_dataset(brand_path)

Generating 200 fake images for D:/VIT Personal/TrustFilterAI/ml/counterfeit_detection/data/raw\Audemars Piguet...


100%|██████████| 200/200 [00:03<00:00, 62.32it/s]


Generating 200 fake images for D:/VIT Personal/TrustFilterAI/ml/counterfeit_detection/data/raw\Breguet...


100%|██████████| 200/200 [00:02<00:00, 76.57it/s]


Generating 200 fake images for D:/VIT Personal/TrustFilterAI/ml/counterfeit_detection/data/raw\Breitling...


...

100%|██████████| 200/200 [00:02<00:00, 83.31it/s]


Generating 200 fake images for D:/VIT Personal/TrustFilterAI/ml/counterfeit_detection/data/raw\Tissot...


100%|██████████| 200/200 [00:02<00:00, 78.10it/s]


Generating 200 fake images for D:/VIT Personal/TrustFilterAI/ml/counterfeit_detection/data/raw\Vacheron Constantin...


100%|██████████| 200/200 [00:02<00:00, 73.45it/s]
